In [ ]:
##########################################################         Incremental Load  #########################################################################


###### <mark>**What we will build**</mark>

**We will create a new demo table with an extra column:
ingestion_ts**

<mark>**Then:**</mark>

**create target table
get max ingestion_ts as watermark
simulate incoming rows with old and new timestamps
filter ingestion_ts > last_watermark
append only new rows
validate inserted rows and new watermark**

###### <mark>**Step 1: Create a timestamp-based demo target**</mark>

In [1]:
spark.sql("DROP TABLE IF EXISTS fact_sales_incremental_ts_demo")

StatementMeta(, 0e6986c3-4818-4868-9ea8-e0545b7f4bd1, 3, Finished, Available, Finished, False)

DataFrame[]

In [2]:
from pyspark.sql.functions import to_timestamp, concat_ws, lit

fact_sales_ts_base_df = (
    spark.table("fact_sales")
    .withColumn(
        "ingestion_ts",
        to_timestamp(
            concat_ws(" ", "order_date", lit("09:00:00"))
        )
    )
)

StatementMeta(, 0e6986c3-4818-4868-9ea8-e0545b7f4bd1, 4, Finished, Available, Finished, False)

###### <mark>**Validate Schema Change**</mark>

In [3]:
fact_sales_ts_base_df.printSchema()
fact_sales_ts_base_df.show(5)

StatementMeta(, 0e6986c3-4818-4868-9ea8-e0545b7f4bd1, 5, Finished, Available, Finished, False)

root
 |-- order_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- product_id: long (nullable = true)
 |-- order_date: date (nullable = true)
 |-- quantity: long (nullable = true)
 |-- line_revenue: double (nullable = true)
 |-- ingestion_ts: timestamp (nullable = true)

+--------+-----------+----------+----------+--------+-----------------+-------------------+
|order_id|customer_id|product_id|order_date|quantity|     line_revenue|       ingestion_ts|
+--------+-----------+----------+----------+--------+-----------------+-------------------+
|  865094|        104|      9474|2024-03-07|       1|           367.67|2024-03-07 09:00:00|
|  865094|        104|     32057|2024-03-07|       5|82.69999999999999|2024-03-07 09:00:00|
|  865094|        104|     37302|2024-03-07|       5|           414.95|2024-03-07 09:00:00|
| 3576055|        104|     27278|2024-04-09|       3|          1000.14|2024-04-09 09:00:00|
| 3576055|        104|      8130|2024-04-09|       1|         

###### <mark>**Save this to a Delta Table**</mark>

In [4]:
fact_sales_ts_base_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("fact_sales_incremental_ts_demo")

StatementMeta(, 0e6986c3-4818-4868-9ea8-e0545b7f4bd1, 6, Finished, Available, Finished, False)

###### <mark>**Step 2: Capture timestamp watermark**</mark>

In [5]:
ts_watermark_row = spark.sql("""
SELECT MAX(ingestion_ts) AS last_loaded_ts
FROM fact_sales_incremental_ts_demo
""").collect()[0]

last_loaded_ts = ts_watermark_row["last_loaded_ts"]

print("Current timestamp watermark:", last_loaded_ts)

StatementMeta(, 0e6986c3-4818-4868-9ea8-e0545b7f4bd1, 7, Finished, Available, Finished, False)

Current timestamp watermark: 2026-03-07 09:00:00


###### <mark>**Step 3: Create incoming batch with intra-day timestamps**</mark>
**This batch will contain:**

- **- one older record**
- **- one same-boundary record**
- **- two new records later on the same day / next day**

In [6]:
from pyspark.sql.types import StructType, StructField, LongType, DateType, DoubleType, TimestampType
from datetime import date, datetime

incoming_orders_ts_data = [
    (999100001, 201, 3001, date(2026, 3, 7), 2, 120.00, datetime(2026, 3, 7, 8, 0, 0)),   # old, ignore
    (999100002, 202, 3002, date(2026, 3, 7), 1,  90.00, datetime(2026, 3, 7, 9, 0, 0)),   # equal boundary, ignore
    (999100003, 203, 3003, date(2026, 3, 7), 4, 300.00, datetime(2026, 3, 7, 12, 0, 0)),  # new, load
    (999100004, 204, 3004, date(2026, 3, 7), 3, 210.00, datetime(2026, 3, 7, 15, 30, 0)), # new, load
    (999100005, 205, 3005, date(2026, 3, 8), 5, 450.00, datetime(2026, 3, 8, 9, 15, 0))   # new, load
]

incoming_orders_ts_schema = StructType([
    StructField("order_id", LongType(), True),
    StructField("customer_id", LongType(), True),
    StructField("product_id", LongType(), True),
    StructField("order_date", DateType(), True),
    StructField("quantity", LongType(), True),
    StructField("line_revenue", DoubleType(), True),
    StructField("ingestion_ts", TimestampType(), True)
])

incoming_orders_ts_df = spark.createDataFrame(
    incoming_orders_ts_data,
    schema=incoming_orders_ts_schema
)

incoming_orders_ts_df.show(truncate=False)

StatementMeta(, 0e6986c3-4818-4868-9ea8-e0545b7f4bd1, 8, Finished, Available, Finished, False)

+---------+-----------+----------+----------+--------+------------+-------------------+
|order_id |customer_id|product_id|order_date|quantity|line_revenue|ingestion_ts       |
+---------+-----------+----------+----------+--------+------------+-------------------+
|999100001|201        |3001      |2026-03-07|2       |120.0       |2026-03-07 08:00:00|
|999100002|202        |3002      |2026-03-07|1       |90.0        |2026-03-07 09:00:00|
|999100003|203        |3003      |2026-03-07|4       |300.0       |2026-03-07 12:00:00|
|999100004|204        |3004      |2026-03-07|3       |210.0       |2026-03-07 15:30:00|
|999100005|205        |3005      |2026-03-08|5       |450.0       |2026-03-08 09:15:00|
+---------+-----------+----------+----------+--------+------------+-------------------+



###### <mark>**Step 4: Filter using timestamp watermark**</mark>

In [7]:
incremental_orders_ts_df = incoming_orders_ts_df.filter(
    incoming_orders_ts_df["ingestion_ts"] > last_loaded_ts
)

incremental_orders_ts_df.show(truncate=False)

StatementMeta(, 0e6986c3-4818-4868-9ea8-e0545b7f4bd1, 9, Finished, Available, Finished, False)

+---------+-----------+----------+----------+--------+------------+-------------------+
|order_id |customer_id|product_id|order_date|quantity|line_revenue|ingestion_ts       |
+---------+-----------+----------+----------+--------+------------+-------------------+
|999100003|203        |3003      |2026-03-07|4       |300.0       |2026-03-07 12:00:00|
|999100004|204        |3004      |2026-03-07|3       |210.0       |2026-03-07 15:30:00|
|999100005|205        |3005      |2026-03-08|5       |450.0       |2026-03-08 09:15:00|
+---------+-----------+----------+----------+--------+------------+-------------------+



<mark>**Why this is important**</mark>

**Notice this:**

**2026-03-07 12:00:00 /
2026-03-07 15:30:00**

**These are on the same date as the watermark day, but still valid new rows.**

**Date-based watermark would have missed this nuance.
Timestamp watermark handles it correctly.**

###### <mark>**Step 5: Append only incremental timestamp rows**</mark>

In [8]:
incremental_orders_ts_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("fact_sales_incremental_ts_demo")

StatementMeta(, 0e6986c3-4818-4868-9ea8-e0545b7f4bd1, 10, Finished, Available, Finished, False)

###### <mark>**Step 6: Validate result**</mark>
**Count check**

**Base table had 24,000,000 rows. ----->>> 
Expected now:
24,000,003**

In [9]:
spark.sql("""
SELECT COUNT(*) AS total_rows
FROM fact_sales_incremental_ts_demo
""").show()

StatementMeta(, 0e6986c3-4818-4868-9ea8-e0545b7f4bd1, 11, Finished, Available, Finished, False)

+----------+
|total_rows|
+----------+
|  24000003|
+----------+



###### <mark>**New watermark check**</mark>

In [10]:
spark.sql("""
SELECT MAX(ingestion_ts) AS new_ts_watermark
FROM fact_sales_incremental_ts_demo
""").show(truncate=False)

StatementMeta(, 0e6986c3-4818-4868-9ea8-e0545b7f4bd1, 12, Finished, Available, Finished, False)

+-------------------+
|new_ts_watermark   |
+-------------------+
|2026-03-08 09:15:00|
+-------------------+



###### <mark>**New rows check**</mark>

In [11]:
spark.sql("""
SELECT *
FROM fact_sales_incremental_ts_demo
WHERE order_id BETWEEN 999100001 AND 999100005
ORDER BY order_id
""").show(truncate=False)

StatementMeta(, 0e6986c3-4818-4868-9ea8-e0545b7f4bd1, 13, Finished, Available, Finished, False)

+---------+-----------+----------+----------+--------+------------+-------------------+
|order_id |customer_id|product_id|order_date|quantity|line_revenue|ingestion_ts       |
+---------+-----------+----------+----------+--------+------------+-------------------+
|999100003|203        |3003      |2026-03-07|4       |300.0       |2026-03-07 12:00:00|
|999100004|204        |3004      |2026-03-07|3       |210.0       |2026-03-07 15:30:00|
|999100005|205        |3005      |2026-03-08|5       |450.0       |2026-03-08 09:15:00|
+---------+-----------+----------+----------+--------+------------+-------------------+



###### <mark>**Timestamp watermark**</mark>

**Better for:**

- multiple loads per day
- more realistic production scheduling
- finer-grained control